# Hippocampus -> Amygdala  Check Dimensionality

Per-trial Factor Analysis dimensionality on cached HFA. Mirrors the upstream `check-data-dimensionality.ipynb` (Binish/Semedo).

**Run after** `hipp_amyg_compute_hfa.ipynb` has produced cached pickles for the band you select via `CONFIG["analysis_band"]`. Switching bands = change that string and re-run from `code-load-hfa`.


## 1. Setup & Config

In [ ]:
# Pin each worker's BLAS to a single thread BEFORE numpy/scipy/sklearn import.
import os
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'BLIS_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_v, '1')

import warnings, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats as scistats

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [ ]:
PROJECT_ROOT = Path('.').resolve()
OUT_DIR = PROJECT_ROOT / 'outputs' / 'hipp_amyg_HFA_CSA_v2'
HFA_CACHE_DIR = OUT_DIR / 'hfa_cache'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Test-mode toggle for the dimensionality analysis.
# True  = fast/noisy (subsample trials, fewer folds, more aggregation).
# False = strict Binish-faithful settings.
TEST_MODE = True

CONFIG = {
    'random_seed': 20260508,

    # --- Region pair (plot labels only; channel selection is in the cache) ---
    'source_region': 'Hippocampus',
    'target_region': 'Amygdala',

    # --- Multi-band selector: which cached band to load ---
    'analysis_band': 'high_gamma',

    # --- Block aggregation along time axis ---
    'aggregate_factor': 10,
    'aggregate_method': 'block_avg',

    # --- Factor Analysis (Binish/Semedo-style) ---
    'fa_n_folds': 10,
    'fa_threshold': 0.95,
    'fa_n_trials_per_session': None,
    'fa_session_n_jobs': 24,    # physical cores; do NOT use -1 (counts hyperthreads)
    'fa_fold_n_jobs': 1,
}

# --- TEST_MODE override ---
if TEST_MODE:
    CONFIG['fa_n_trials_per_session'] = 30
    CONFIG['fa_n_folds'] = 5
    CONFIG['aggregate_factor'] = 20
    print('[TEST_MODE override applied] '
          f"trials/session={CONFIG['fa_n_trials_per_session']}, "
          f"n_folds={CONFIG['fa_n_folds']}, "
          f"aggregate_factor={CONFIG['aggregate_factor']}")

print(f"Source -> Target: {CONFIG['source_region']} -> {CONFIG['target_region']}")
print(f"Analysis band: {CONFIG['analysis_band']}")
print(f"Cache dir: {HFA_CACHE_DIR}/{CONFIG['analysis_band']}")
print(f"FA parallelism: session_n_jobs={CONFIG['fa_session_n_jobs']}, "
      f"fold_n_jobs={CONFIG['fa_fold_n_jobs']}")


## 2. Helpers

In [ ]:
# --- Trial grouping ---
def group_trials_by_load(trial_table):
    if 'loads' not in trial_table.columns:
        raise ValueError("trial_table needs a 'loads' column")
    load_values = sorted(trial_table['loads'].dropna().unique())
    groups, labels, counts = [], [], []
    for lv in load_values:
        idx = np.where(trial_table['loads'].values == lv)[0]
        groups.append(idx)
        labels.append(f'Load {int(lv) if lv == int(lv) else lv}')
        counts.append(len(idx))
    return {'groups': groups, 'group_labels': labels, 'group_counts': counts,
            'load_values': load_values,
            'summary': '\n'.join([f'Load grouping: {len(trial_table)} trials'] +
                                  [f'  {l}: {c}' for l, c in zip(labels, counts)])}


def group_trials_by_first_stim_category(trial_table, keep=(1, 2, 3, 4, 5)):
    if 'PicIDs_Encoding1' not in trial_table.columns:
        raise ValueError("trial_table needs a 'PicIDs_Encoding1' column")
    pic_ids = trial_table['PicIDs_Encoding1'].fillna(0).astype(int)
    cats = pic_ids // 100
    keep = list(keep)
    groups = [np.where(cats.values == c)[0] for c in keep]
    labels = [f'cat {c}' for c in keep]
    counts = [len(g) for g in groups]
    dropped = ~cats.isin(keep) | (pic_ids == 0)
    return {'groups': groups, 'group_labels': labels, 'group_counts': counts,
            'load_values': keep, 'dropped_trials': np.where(dropped.values)[0],
            'summary': '\n'.join([f'Category grouping: {len(trial_table)} trials, keep={tuple(keep)}'] +
                                  [f'  {l}: {c}' for l, c in zip(labels, counts)] +
                                  [f'  [dropped] {int(dropped.sum())} trials'])}


def get_trial_groups(trial_table, mode='category', keep=(1, 2, 3, 4, 5)):
    if mode == 'load':
        return group_trials_by_load(trial_table)
    if mode == 'category':
        return group_trials_by_first_stim_category(trial_table, keep=keep)
    raise ValueError(f"unknown trial_grouping mode: {mode!r}")


# --- Block aggregation ---
def _aggregate(X, factor, method='block_avg'):
    if factor is None or factor <= 1:
        return X
    if method == 'decimate':
        return X[::factor]
    if method == 'block_avg':
        n = (X.shape[0] // factor) * factor
        return X if n == 0 else X[:n].reshape(-1, factor, X.shape[1]).mean(axis=1)
    raise ValueError(f'unknown method {method!r}')


print('grouping + aggregation helpers defined')

In [ ]:
# --- Factor Analysis (sklearn FA + Semedo/Binish-style model selection)
#     Mirrors the FA pipeline in nehabinish/pfc-m1-communication-subspace
#     (src/dimreduction_utils.py): K-fold CV log-likelihood across a q grid
#     including q=0 (diagonal Gaussian), pick qMax = argmax mean CV LL, refit
#     FA at qMax on full X, then take the smallest q whose cumulative shared
#     variance from eigvals of L @ L.T exceeds 0.95.
#     EM/log-likelihood via sklearn.decomposition.FactorAnalysis.
#     Folds parallelized via joblib (matches Binish's CrossValFa(parallel=True)).
from sklearn.decomposition import FactorAnalysis
from sklearn.model_selection import KFold
from sklearn.exceptions import ConvergenceWarning
from joblib import Parallel, delayed

warnings.filterwarnings('ignore', category=ConvergenceWarning, module='sklearn')

_FA_TOL = 1e-3
_FA_MAX_ITER = 1500


def _diagonal_gaussian_score(Xtrain, Xtest):
    """Per-sample log-likelihood of Xtest under N(mean(Xtrain), diag(var(Xtrain)))."""
    mean = Xtrain.mean(axis=0)
    var = Xtrain.var(axis=0, ddof=0) + 1e-12
    Xc = Xtest - mean
    n = Xtest.shape[0]
    return -0.5 * (np.sum(np.log(2.0 * np.pi * var)) + (Xc ** 2 / var).sum() / max(n, 1))


def _fa_one_fold(Xtr, Xte, q_grid, seed, tol, max_iter):
    """Per-q test log-likelihood for one fold. Top-level so joblib can pickle it."""
    cv_ll_row = np.full(len(q_grid), np.nan)
    for qi, q in enumerate(q_grid):
        try:
            if q == 0:
                cv_ll_row[qi] = _diagonal_gaussian_score(Xtr, Xte)
            else:
                fa = FactorAnalysis(n_components=int(q), random_state=seed,
                                     tol=tol, max_iter=max_iter)
                fa.fit(Xtr)
                cv_ll_row[qi] = fa.score(Xte)
        except Exception:
            pass
    return cv_ll_row


def fa_cross_validate(X, q_grid, n_folds=10, rng=None, fold_n_jobs=1):
    """K-fold CV. Returns (cvLoss, cvLogLike, expvar). See helper docstring."""
    rng = rng if rng is not None else np.random.default_rng(0)
    q_grid = np.sort(np.asarray(q_grid, dtype=int))
    seed = int(rng.integers(0, 2**31 - 1))
    N, p = X.shape

    # Binish uses KFold default (shuffle=False) — match that here.
    kf = KFold(n_splits=n_folds)
    splits = [(X[tr], X[te]) for tr, te in kf.split(X)]

    if fold_n_jobs == 1:
        cv_ll = np.array([
            _fa_one_fold(Xtr, Xte, q_grid, seed, _FA_TOL, _FA_MAX_ITER)
            for Xtr, Xte in splits
        ])
    else:
        # Threading backend: low overhead per call, avoids nested-process
        # oversubscription when the outer session loop is already loky-based.
        cv_ll = np.array(Parallel(n_jobs=fold_n_jobs, backend='threading')(
            delayed(_fa_one_fold)(Xtr, Xte, q_grid, seed, _FA_TOL, _FA_MAX_ITER)
            for Xtr, Xte in splits
        ))

    mean_ll = np.nanmean(cv_ll, axis=0)
    if not np.any(np.isfinite(mean_ll)):
        return np.full(len(q_grid), np.nan), cv_ll, np.zeros(p)
    qMax = int(q_grid[np.nanargmax(mean_ll)])
    if qMax == 0:
        return np.full(len(q_grid), np.nan), cv_ll, np.zeros(p)

    fa_full = FactorAnalysis(n_components=qMax, random_state=seed,
                              tol=_FA_TOL, max_iter=_FA_MAX_ITER).fit(X)
    L = fa_full.components_.T   # (p, qMax)
    eigs = np.sort(np.linalg.eigvalsh(L @ L.T))[::-1]
    eigs = np.clip(eigs, 0.0, None)
    total = float(eigs.sum())
    if total <= 0:
        return np.full(len(q_grid), np.nan), cv_ll, np.zeros(p)
    expvar = eigs / total
    cum_loss = 1.0 - np.cumsum(eigs) / total

    cvLoss = np.full(len(q_grid), np.nan)
    for i, qi in enumerate(q_grid):
        if qi == 0:
            cvLoss[i] = 1.0
        elif qi - 1 < cum_loss.size:
            cvLoss[i] = float(cum_loss[qi - 1])
        else:
            cvLoss[i] = 0.0
    return cvLoss, cv_ll, expvar


def fa_select_dim(cvLoss, q_grid, threshold=0.95):
    """Smallest q with cumulative shared variance >= threshold."""
    cvLoss = np.asarray(cvLoss)
    q_grid = np.asarray(q_grid, dtype=int)
    if np.any(np.isnan(cvLoss)):
        return 0
    above = (1.0 - cvLoss) > threshold
    if not above.any():
        return int(q_grid[-1])
    return int(q_grid[int(np.argmax(above))])


def estimate_dim_per_trial(tensor, q_grid=None, n_folds=10, downsample=10,
                           aggregate_method='block_avg', rng=None,
                           threshold=0.95, fold_n_jobs=1):
    """Per-trial FA dimensionality. tensor: (n_trials, n_chan, n_t)."""
    rng = rng if rng is not None else np.random.default_rng(0)
    n_trials, n_chan, _ = tensor.shape
    if q_grid is None:
        q_grid = np.arange(0, n_chan + 1)        # Binish: np.arange(0, nfeatures + 1)
    q_grid = np.asarray(q_grid, dtype=int)
    dims = np.full(n_trials, np.nan)
    expvars = [None] * n_trials
    for ti in range(n_trials):
        X = tensor[ti].T.astype(np.float64)
        X = _aggregate(X, downsample, method=aggregate_method)
        if X.shape[0] < (X.shape[1] + n_folds + 1):
            continue
        try:
            cvLoss, _, expvar = fa_cross_validate(X, q_grid, n_folds=n_folds,
                                                  rng=rng, fold_n_jobs=fold_n_jobs)
            dims[ti] = float(fa_select_dim(cvLoss, q_grid, threshold=threshold))
            expvars[ti] = expvar
        except Exception:
            continue
    return {'dim': dims, 'expvar': expvars, 'q_grid': q_grid}


print('FA helpers defined (sklearn FA + joblib folds + Binish-style 95% selection)')


def _dim_one(session_result, cfg):
    """Per-session FA on encoding-locked src/tgt tensors. Top-level so loky can pickle it."""
    if session_result is None:
        return None
    enc = session_result['event_tensors'].get('encoding1')
    if enc is None or enc['src'].shape[0] == 0:
        return None
    rng_local = np.random.default_rng(cfg['random_seed'] + hash(session_result['session']) % 10000)

    n_trials = enc['src'].shape[0]
    take = cfg['fa_n_trials_per_session']
    if take is not None and take < n_trials:
        idx = np.sort(rng_local.choice(n_trials, take, replace=False))
        src_t = enc['src'][idx]; tgt_t = enc['tgt'][idx]
    else:
        src_t = enc['src']; tgt_t = enc['tgt']

    src_res = estimate_dim_per_trial(
        src_t, n_folds=cfg['fa_n_folds'], downsample=cfg['aggregate_factor'],
        aggregate_method=cfg['aggregate_method'], threshold=cfg['fa_threshold'],
        rng=rng_local, fold_n_jobs=cfg['fa_fold_n_jobs'],
    )
    tgt_res = estimate_dim_per_trial(
        tgt_t, n_folds=cfg['fa_n_folds'], downsample=cfg['aggregate_factor'],
        aggregate_method=cfg['aggregate_method'], threshold=cfg['fa_threshold'],
        rng=rng_local, fold_n_jobs=cfg['fa_fold_n_jobs'],
    )
    return {
        'subject': session_result['subject'], 'session': session_result['session'],
        'src_dim_mean': float(np.nanmean(src_res['dim'])),
        'tgt_dim_mean': float(np.nanmean(tgt_res['dim'])),
        'src_dim_per_trial': src_res['dim'],
        'tgt_dim_per_trial': tgt_res['dim'],
        'src_n_chan': src_t.shape[1], 'tgt_n_chan': tgt_t.shape[1],
        'n_trials_used': src_t.shape[0],
    }



## 3. Load HFA from disk

Reads every `hfa_*.pkl` file in `<HFA_CACHE_DIR>/<analysis_band>/` into `all_session_hfa`, sorted by subject/session.


In [ ]:
import pickle
from pathlib import Path

band = CONFIG['analysis_band']
band_dir = Path(HFA_CACHE_DIR) / band
cache_files = sorted(band_dir.glob('hfa_*.pkl'))
print(f'Loading band {band!r} from {band_dir}')
print(f'Found {len(cache_files)} cached pickles')

if not cache_files:
    raise FileNotFoundError(
        f'No cache files in {band_dir}. Run hipp_amyg_compute_hfa.ipynb to generate '
        f'the {band!r} band first, or change CONFIG[\'analysis_band\'] to one that exists.'
    )

all_session_hfa = []
for fp in cache_files:
    try:
        with open(fp, 'rb') as f:
            r = pickle.load(f)
        if r is None:
            continue
        all_session_hfa.append(r)
    except Exception as e:
        print(f'  [skip] {fp.name}: {e}')

all_session_hfa.sort(key=lambda r: (r['subject'], r['session']))
print(f'Loaded {len(all_session_hfa)} sessions for band {band!r}:')
for r in all_session_hfa:
    print(f'  {r["subject"]}/{r["session"]}  '
          f'src={len(r["src_chans"])}  tgt={len(r["tgt_chans"])}')


## 4. Dimensionality Analysis (Binish/Semedo-style cross-validated FA)

Per-trial Factor Analysis on the **encoding-locked** tensor `(n_trials, n_chan, n_t)`. For each trial: transpose to `[timepoints, channels]`, block-aggregate, sweep `q in {0, 1, ..., n_chan}` with K-fold CV, pick `qMax = argmax mean CV log-likelihood`, refit FA at `qMax` on full X, and take **qOpt = smallest q with cumulative shared variance >= 0.95**. Per-session: report `mean(qOpt)` over trials for src and tgt, paired Wilcoxon across sessions.


### 6a. Single-session demo (run before the full sweep)

Times one session end-to-end so you can sanity-check the pipeline and estimate full-sweep wall-clock before kicking off all 28.

In [ ]:
# Demo: run ONE session through the per-trial FA pipeline with progress prints,
# then plot the Binish/Semedo-style cumulative-shared-variance curves and the
# per-trial qOpt distribution. Mirrors _dim_one but loops trials manually so we
# can capture per-trial expvar for the plot.
DEMO_SUBJECT = 'sub-15'
DEMO_SESSION = 'sub-15_ses-1_ecephys+image'
PROGRESS_EVERY = 10
N_FACTORS_SHOW = 10   # x-axis range for the cumulative-variance plot

demo_session = next((r for r in all_session_hfa
                      if r['subject'] == DEMO_SUBJECT and r['session'] == DEMO_SESSION),
                     None)
if demo_session is None:
    available = [(r['subject'], r['session']) for r in all_session_hfa]
    raise RuntimeError(f'{DEMO_SUBJECT}/{DEMO_SESSION} not found in all_session_hfa.\n'
                       f'Available: {available}')

enc = demo_session['event_tensors']['encoding1']
n_trials_total = enc['src'].shape[0]
print(f'Demo session: {DEMO_SUBJECT}/{DEMO_SESSION}  '
      f'(src={len(demo_session["src_chans"])} ch, tgt={len(demo_session["tgt_chans"])} ch, '
      f'trials={n_trials_total})')

rng_local = np.random.default_rng(CONFIG['random_seed'] + hash(DEMO_SESSION) % 10000)
take = CONFIG['fa_n_trials_per_session']
if take is not None and take < n_trials_total:
    idx = np.sort(rng_local.choice(n_trials_total, take, replace=False))
    src_t = enc['src'][idx]; tgt_t = enc['tgt'][idx]
else:
    src_t = enc['src']; tgt_t = enc['tgt']


def _run_region_verbose(label, tensor):
    """Per-trial FA with progress prints. Returns (dims_array, list_of_expvar)."""
    n_trials, n_chan, _ = tensor.shape
    q_grid = np.arange(0, n_chan + 1)
    dims = np.full(n_trials, np.nan)
    expvars = []
    n_skipped = n_failed = 0
    print(f'  [{label}] {n_trials} trials, {n_chan} channels, q grid 0..{n_chan}')
    t_region = time.time()
    for ti in range(n_trials):
        X = tensor[ti].T.astype(np.float64)
        X = _aggregate(X, CONFIG['aggregate_factor'], method=CONFIG['aggregate_method'])
        if X.shape[0] < (X.shape[1] + CONFIG['fa_n_folds'] + 1):
            n_skipped += 1
            continue
        try:
            cvLoss, _, expvar = fa_cross_validate(
                X, q_grid, n_folds=CONFIG['fa_n_folds'],
                rng=rng_local, fold_n_jobs=CONFIG['fa_fold_n_jobs'],
            )
            dims[ti] = float(fa_select_dim(cvLoss, q_grid,
                                            threshold=CONFIG['fa_threshold']))
            if expvar is not None and len(expvar) > 0 and np.any(expvar > 0):
                expvars.append(np.asarray(expvar))
        except Exception:
            n_failed += 1
            continue
        done = ti + 1
        if done % PROGRESS_EVERY == 0 or done == n_trials:
            elapsed = time.time() - t_region
            rate = done / elapsed if elapsed > 0 else 0
            eta = (n_trials - done) / rate if rate > 0 else float('nan')
            print(f'    [{label}] trial {done:>3}/{n_trials}  '
                  f'mean qOpt so far {np.nanmean(dims):.2f}  '
                  f'elapsed {elapsed:.1f}s  ETA {eta:.1f}s')
    print(f'  [{label}] done in {time.time()-t_region:.1f}s '
          f'(mean qOpt={np.nanmean(dims):.2f}, '
          f'skipped={n_skipped}, failed={n_failed})')
    return dims, expvars


t0 = time.time()
src_dims, src_ev = _run_region_verbose('src', src_t)
tgt_dims, tgt_ev = _run_region_verbose('tgt', tgt_t)
total = time.time() - t0

print(f'\nTotal elapsed: {total:.1f}s ({total/60:.1f} min)')
print(f'  src qOpt mean = {np.nanmean(src_dims):.2f}  '
      f'(per-trial range {np.nanmin(src_dims):.0f}\u2013{np.nanmax(src_dims):.0f})')
print(f'  tgt qOpt mean = {np.nanmean(tgt_dims):.2f}  '
      f'(per-trial range {np.nanmin(tgt_dims):.0f}\u2013{np.nanmax(tgt_dims):.0f})')


def _cumvar_curve(expvars, n_show):
    """Stack per-trial cumulative shared-variance curves, padded/truncated to n_show.

    Each entry of expvars is a length-p array of normalised eigvals of L@L.T.
    Returns (mean across trials, SE across trials) of shape (n_show,)."""
    if not expvars:
        return None, None
    rows = []
    for ev in expvars:
        cum = np.cumsum(ev)
        if len(cum) >= n_show:
            rows.append(cum[:n_show])
        else:
            pad = np.full(n_show, cum[-1] if len(cum) else 0.0)
            pad[:len(cum)] = cum
            rows.append(pad)
    arr = np.array(rows)
    n = max(arr.shape[0], 1)
    return arr.mean(axis=0), arr.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.zeros(n_show)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- (a) Cumulative shared variance vs #factors ---
ax = axes[0]
xs = np.arange(1, N_FACTORS_SHOW + 1)
for label, ev_list, color in [(CONFIG['source_region'], src_ev, '#4C72B0'),
                                (CONFIG['target_region'], tgt_ev, '#DD8452')]:
    mean, se = _cumvar_curve(ev_list, N_FACTORS_SHOW)
    if mean is None:
        continue
    ax.plot(xs, mean, '-o', color=color, lw=2, markersize=5,
            label=f'{label} (n={len(ev_list)} trials)')
    ax.fill_between(xs, mean - se, mean + se, color=color, alpha=0.2)
ax.axhline(CONFIG['fa_threshold'], color='k', ls='--', lw=1, alpha=0.6,
           label=f'{int(CONFIG["fa_threshold"]*100)}% threshold')
ax.set_xticks(xs)
ax.set_xlabel('Number of factors')
ax.set_ylabel('Cumulative shared variance')
ax.set_title(f'FA cumulative variance \u2014 '
             f'{DEMO_SUBJECT}/{DEMO_SESSION.split("_ecephys")[0]}')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.02)

# --- (b) Per-trial qOpt distribution ---
ax = axes[1]
src_d = src_dims[~np.isnan(src_dims)]
tgt_d = tgt_dims[~np.isnan(tgt_dims)]
if len(src_d) and len(tgt_d):
    parts = ax.violinplot([src_d, tgt_d], positions=[1, 2], widths=0.7,
                           showmeans=True, showmedians=True)
    for body, color in zip(parts['bodies'], ['#4C72B0', '#DD8452']):
        body.set_facecolor(color); body.set_alpha(0.6)
    ax.scatter(np.full(len(src_d), 1) + rng_local.uniform(-0.08, 0.08, len(src_d)),
                src_d, color='#4C72B0', alpha=0.4, s=20)
    ax.scatter(np.full(len(tgt_d), 2) + rng_local.uniform(-0.08, 0.08, len(tgt_d)),
                tgt_d, color='#DD8452', alpha=0.4, s=20)
ax.set_xticks([1, 2])
ax.set_xticklabels([CONFIG['source_region'], CONFIG['target_region']])
ax.set_ylabel('Per-trial qOpt')
ax.set_title(f'Per-trial qOpt (src n={len(src_d)}, tgt n={len(tgt_d)})')

plt.tight_layout()
plt.savefig(OUT_DIR / 'dim_demo_cumvar.png', dpi=130, bbox_inches='tight')
plt.show()

_n_sess = len(all_session_hfa)
_n_workers = CONFIG['fa_session_n_jobs']
_batches = int(np.ceil(_n_sess / _n_workers))
print(f'\nFull-sweep wall-clock estimate (this-session pace): '
      f'~{total * _batches / 60:.0f} min  '
      f'({_batches} batch(es) of up to {_n_workers} parallel sessions)')
print('If timing looks acceptable, run code-fa-runner to do all sessions.')


In [ ]:
# Outer parallelism: loky processes (real CPU cores, sidesteps the GIL).
# Inner parallelism via cfg['fa_fold_n_jobs']. Keep
# fa_session_n_jobs * fa_fold_n_jobs <= n_physical_cores.
t0 = time.time()
session_n_jobs = CONFIG.get('fa_session_n_jobs', 24)
print(f'Dim sweep: {len(all_session_hfa)} sessions, '
      f'session_n_jobs={session_n_jobs}, fold_n_jobs={CONFIG["fa_fold_n_jobs"]}')

dim_results = Parallel(n_jobs=session_n_jobs, backend='loky', verbose=10)(
    delayed(_dim_one)(r, CONFIG) for r in all_session_hfa
)
dim_rows = [r for r in dim_results if r is not None]

for r in dim_rows:
    print(f'  [done] {r["subject"]}/{r["session"]}: '
          f'src qOpt={r["src_dim_mean"]:.2f}, tgt qOpt={r["tgt_dim_mean"]:.2f}')

dim_df = pd.DataFrame(dim_rows)
dim_df.to_csv(OUT_DIR / 'dim_summary.csv', index=False)
print(f'\nDone: {len(dim_df)} sessions in {time.time()-t0:.0f}s -> {OUT_DIR/"dim_summary.csv"}')
dim_df.head()


In [ ]:
# Group-level paired test + plots
src_col_name = f'src_dim_mean'
tgt_col_name = f'tgt_dim_mean'
paired = dim_df.dropna(subset=[src_col_name, tgt_col_name])

if len(paired) >= 5:
    stat, p = scistats.wilcoxon(paired[src_col_name], paired[tgt_col_name], zero_method='wilcox')
    print(f'Group-level paired Wilcoxon (session-mean dim95):')
    print(f'  n={len(paired)}, W={stat:.1f}, p={p:.4g}')
    print(f'  median {CONFIG["source_region"]} = {paired[src_col_name].median():.2f},  '
          f'median {CONFIG["target_region"]} = {paired[tgt_col_name].median():.2f}')

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.boxplot(
    [paired[src_col_name], paired[tgt_col_name]],
    labels=[CONFIG['source_region'], CONFIG['target_region']],
    showmeans=True, patch_artist=True,
)
for patch, color in zip(parts['boxes'], ['#4C72B0', '#DD8452']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
for _, r in paired.iterrows():
    ax.plot([1, 2], [r[src_col_name], r[tgt_col_name]], 'k-', alpha=0.25, lw=0.7)
    ax.plot([1, 2], [r[src_col_name], r[tgt_col_name]], 'ko', alpha=0.5, markersize=4)
ax.set_ylabel('Session-mean dim95')
ax.set_title(f'Group-level dimensionality (n={len(paired)} sessions)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'dim_group.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Trial-level stats — every per-trial qOpt is a sample, not just the session mean.
# Three views, complementing the session-level Wilcoxon above:
#   (1) trial-level qOpt distribution per region (descriptive violin)
#   (2) within-session paired Wilcoxon across trials -> one p per session
#   (3) linear mixed-effects: qOpt ~ region + (1|session)
#       Trial-level inference that respects trials-within-session non-independence.

import statsmodels.formula.api as smf

# Long-format: one row per (session, trial, region)
long_rows = []
for _, r in dim_df.iterrows():
    for kind, arr in [('src', r['src_dim_per_trial']),
                       ('tgt', r['tgt_dim_per_trial'])]:
        if not hasattr(arr, '__iter__'):
            continue
        for ti, d in enumerate(arr):
            if np.isfinite(d):
                long_rows.append({
                    'subject': r['subject'], 'session': r['session'],
                    'trial': ti, 'region': kind, 'dim': float(d),
                })
long_df = pd.DataFrame(long_rows)
print(f'Trial-level rows: {len(long_df)}  '
      f'({long_df["session"].nunique()} sessions, '
      f'{long_df["subject"].nunique()} subjects)')

# --- (1) Trial-level distribution ---
src_vals = long_df.query('region == "src"')['dim'].values
tgt_vals = long_df.query('region == "tgt"')['dim'].values
fig, ax = plt.subplots(figsize=(8, 5))
parts = ax.violinplot([src_vals, tgt_vals], positions=[1, 2], widths=0.7,
                       showmeans=True, showmedians=True)
for body, color in zip(parts['bodies'], ['#4C72B0', '#DD8452']):
    body.set_facecolor(color); body.set_alpha(0.6)
ax.set_xticks([1, 2])
ax.set_xticklabels([CONFIG['source_region'], CONFIG['target_region']])
ax.set_ylabel('Per-trial qOpt')
ax.set_title(f'Trial-level qOpt  (src n={len(src_vals)}, tgt n={len(tgt_vals)})')
plt.tight_layout()
plt.savefig(OUT_DIR / 'dim_trial_violin.png', dpi=130, bbox_inches='tight')
plt.show()

# --- (2) Within-session paired Wilcoxon ---
print('\n=== Within-session paired Wilcoxon (per-trial src vs tgt qOpt) ===')
sess_rows = []
for sess, g in long_df.groupby('session'):
    src = g.query('region == "src"').sort_values('trial')['dim'].values
    tgt = g.query('region == "tgt"').sort_values('trial')['dim'].values
    n = min(len(src), len(tgt))
    if n < 10:
        continue
    src, tgt = src[:n], tgt[:n]
    if np.allclose(src, tgt):
        continue
    try:
        stat, p = scistats.wilcoxon(src, tgt, zero_method='wilcox')
        sess_rows.append({'session': sess, 'n_trials': n,
                          'W': float(stat), 'p': float(p),
                          'src_median': float(np.median(src)),
                          'tgt_median': float(np.median(tgt)),
                          'diff_median': float(np.median(src - tgt))})
    except Exception:
        continue
sess_p_df = pd.DataFrame(sess_rows)
if len(sess_p_df):
    n_tests = len(sess_p_df)
    sess_p_df['p_bonf'] = np.minimum(sess_p_df['p'] * n_tests, 1.0)
    print(f'  sessions tested: {n_tests}')
    print(f'  p<0.05 uncorrected:  {(sess_p_df["p"] < 0.05).sum()}')
    print(f'  p<0.05 Bonferroni:    {(sess_p_df["p_bonf"] < 0.05).sum()}')
    sess_p_df.to_csv(OUT_DIR / 'dim_session_p_values.csv', index=False)

# --- (3) Linear mixed-effects model ---
print('\n=== Linear mixed-effects: dim ~ region + (1|session) ===')
try:
    md = smf.mixedlm('dim ~ C(region, Treatment("src"))',
                      long_df, groups=long_df['session'])
    fit = md.fit(method='lbfgs', reml=True)
    coef_name = 'C(region, Treatment("src"))[T.tgt]'
    coef = fit.params[coef_name]
    se = fit.bse[coef_name]
    pval = fit.pvalues[coef_name]
    re_sd = float(np.sqrt(fit.cov_re.iloc[0, 0]))
    print(f'  effect of region (tgt - src): {coef:+.3f}  SE={se:.3f}  p={pval:.4g}')
    print(f'  random-intercept SD across sessions: {re_sd:.3f}')
    print()
    print(fit.summary())
except Exception as e:
    print(f'  LMM failed: {e}')
